In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
from src.data_loader import read_csv_from_blob

# Read the last 14 days of actual DAM prices
dam = read_csv_from_blob("sagreekdamdevweu", "processed", "dam_prices/full.csv")
recent = dam[dam.index >= dam.index.max() - pd.Timedelta(days=14)]

# Look at the 19:00 UTC hour each day (the hour we predicted high)
hour_19 = recent[recent.index.hour == 19]
print("Recent 19:00 UTC prices:")
print(hour_19.tail(14))

# Compare to what we predicted: 346
print(f"\nRecent average:  {hour_19['price_eur_mwh'].mean():.1f}")
print(f"Recent max:      {hour_19['price_eur_mwh'].max():.1f}")
print(f"Recent min:      {hour_19['price_eur_mwh'].min():.1f}")
print(f"Predicted (May 9): 346")

Recent 19:00 UTC prices:
                           price_eur_mwh
2026-04-27 19:00:00+00:00       153.7550
2026-04-28 19:00:00+00:00       117.1325
2026-04-29 19:00:00+00:00       172.9700
2026-04-30 19:00:00+00:00       136.0025
2026-05-01 19:00:00+00:00        79.5125
2026-05-02 19:00:00+00:00       123.9375
2026-05-03 19:00:00+00:00       135.6075
2026-05-04 19:00:00+00:00       219.3550
2026-05-05 19:00:00+00:00       157.1175
2026-05-06 19:00:00+00:00       159.5050
2026-05-07 19:00:00+00:00       122.9725
2026-05-08 19:00:00+00:00       156.3450
2026-05-09 19:00:00+00:00       185.7625
2026-05-10 19:00:00+00:00       129.4700

Recent average:  146.4
Recent max:      219.4
Recent min:      79.5
Predicted (May 9): 346


In [3]:
# Read updated DAM (after May 9 data became available)
dam = read_csv_from_blob("sagreekdamdevweu", "processed", "dam_prices/full.csv")
may_9 = dam.loc["2026-05-09":"2026-05-10"]  # all of May 9
print(may_9)

                           price_eur_mwh
2026-05-09 00:00:00+00:00       168.5400
2026-05-09 01:00:00+00:00       147.2000
2026-05-09 02:00:00+00:00       173.5400
2026-05-09 03:00:00+00:00       172.7900
2026-05-09 04:00:00+00:00       146.5125
2026-05-09 05:00:00+00:00       123.3950
2026-05-09 06:00:00+00:00       106.0150
2026-05-09 07:00:00+00:00        33.2900
2026-05-09 08:00:00+00:00         0.0000
2026-05-09 09:00:00+00:00         0.0025
2026-05-09 10:00:00+00:00         0.0000
2026-05-09 11:00:00+00:00         0.0000
2026-05-09 12:00:00+00:00         0.0000
2026-05-09 13:00:00+00:00         0.0000
2026-05-09 14:00:00+00:00        29.9700
2026-05-09 15:00:00+00:00       101.7750
2026-05-09 16:00:00+00:00       146.3625
2026-05-09 17:00:00+00:00       187.5400
2026-05-09 18:00:00+00:00       216.9450
2026-05-09 19:00:00+00:00       185.7625
2026-05-09 20:00:00+00:00       164.8000
2026-05-09 21:00:00+00:00       174.3050
2026-05-09 22:00:00+00:00       162.0250
2026-05-09 23:00

In [4]:
model_path = Path("C:\\dev\\greek-dam-forecasting\\outputs\\model_bundle.pkl")

import pickle
with open(model_path, "rb") as f:
    result = pickle.load(f)

# Feature importance for h=19
import pandas as pd
model = result.models[19]
imp = pd.DataFrame({
    "feature": result.feature_names,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
print(imp.head(20))

                    feature  importance
37       target_net_load_mw         667
34  target_load_forecast_mw         483
36      target_wind_onshore         468
21            tr_lag_d2_k+1         434
19            tr_lag_d2_k-1         425
22            tr_lag_d7_k-1         422
23            tr_lag_d7_k+0         397
14          ft_wind_onshore         394
20            tr_lag_d2_k+0         385
10              ft_max_168h         376
5                ft_std_24h         364
12      ft_load_forecast_mw         363
9               ft_std_168h         356
4               ft_mean_24h         352
8              ft_mean_168h         344
24            tr_lag_d7_k+1         318
6                ft_max_24h         317
15           ft_net_load_mw         310
13                 ft_solar         283
11              ft_min_168h         271


In [5]:
from src.features import build_supervised_dataset
from src.dataset import load_combined_dataset

prices, exog = load_combined_dataset(
    storage_account="sagreekdamdevweu",
    join="outer",  # include tomorrow's exog
)

X, y, meta = build_supervised_dataset(
    prices, exog=exog,
    gate_closure_hour=12,
    horizons=tuple(range(0, 24)),
)

# Find the forecast for May 9 at 19:00 UTC
target_ts = pd.Timestamp("2026-05-09 19:00:00", tz="UTC")
mask = meta["target_time"] == target_ts
features = X[mask]
print("Features for the predicted hour:")
print(features.T)  # transpose for readability

Features for the predicted hour:
                               24478
ft_price_minus_0h          68.612500
ft_price_minus_1h          62.912500
ft_price_minus_2h         103.507500
ft_price_minus_3h          94.867500
ft_mean_24h               121.854375
ft_std_24h                 23.888890
ft_max_24h                160.512500
ft_min_24h                 62.912500
ft_mean_168h               92.526622
ft_std_168h                63.946254
ft_max_168h               255.587500
ft_min_168h                -9.992500
ft_load_forecast_mw      5455.000000
ft_solar                 2750.000000
ft_wind_onshore          1645.000000
ft_net_load_mw           1060.000000
tr_lag_d1_k-1                    NaN
tr_lag_d1_k+0                    NaN
tr_lag_d1_k+1                    NaN
tr_lag_d2_k-1             130.075000
tr_lag_d2_k+0             122.972500
tr_lag_d2_k+1             119.335000
tr_lag_d7_k-1             132.332500
tr_lag_d7_k+0             123.937500
tr_lag_d7_k+1             129.562500
targe

In [6]:
import pandas as pd
from src.data_loader import read_csv_from_blob

# Read full DAM history
dam = read_csv_from_blob("sagreekdamdevweu", "processed", "dam_prices/full.csv")
dam.index = pd.to_datetime(dam.index, utc=True)

# Was May 9 actuals included?
may_9 = dam[(dam.index >= "2026-05-09") & (dam.index < "2026-05-10")]
print(f"May 9 actual values: {len(may_9)} rows")

if len(may_9) == 24:
    # Hour-by-hour comparison
    predicted = pd.Series(
        [61, 56, 58, 41, 77, 71, 56, 80, 58, 67, 34, 33, 36, 37, 107, 92,
         121, 148, 191, 346, 202, 149, 108, 67],
        index=may_9.index,
    )
    
    comparison = pd.DataFrame({
        "predicted": predicted.values,
        "actual": may_9["price_eur_mwh"].values,
    })
    comparison["error"] = comparison["actual"] - comparison["predicted"]
    comparison["abs_error"] = comparison["error"].abs()
    print(comparison)
    print(f"\nMAE: {comparison['abs_error'].mean():.2f}")
    print(f"Worst hour: {comparison['abs_error'].idxmax()}")
else:
    # Look at recent evening peaks
    eve_peaks = dam[dam.index.hour.isin([18, 19, 20])]
    recent = eve_peaks[eve_peaks.index >= "2026-05-01"]
    print(f"\nRecent evening peaks (last 9 days):")
    print(recent.tail(20))

May 9 actual values: 24 rows
    predicted    actual     error  abs_error
0          61  168.5400  107.5400   107.5400
1          56  147.2000   91.2000    91.2000
2          58  173.5400  115.5400   115.5400
3          41  172.7900  131.7900   131.7900
4          77  146.5125   69.5125    69.5125
5          71  123.3950   52.3950    52.3950
6          56  106.0150   50.0150    50.0150
7          80   33.2900  -46.7100    46.7100
8          58    0.0000  -58.0000    58.0000
9          67    0.0025  -66.9975    66.9975
10         34    0.0000  -34.0000    34.0000
11         33    0.0000  -33.0000    33.0000
12         36    0.0000  -36.0000    36.0000
13         37    0.0000  -37.0000    37.0000
14        107   29.9700  -77.0300    77.0300
15         92  101.7750    9.7750     9.7750
16        121  146.3625   25.3625    25.3625
17        148  187.5400   39.5400    39.5400
18        191  216.9450   25.9450    25.9450
19        346  185.7625 -160.2375   160.2375
20        202  164.8000  -

In [7]:
from src.data_loader import read_csv_from_blob
import pandas as pd

dam = read_csv_from_blob("sagreekdamdevweu", "processed", "dam_prices/full.csv")
dam.index = pd.to_datetime(dam.index, utc=True)

# Look at May 9 in detail
may_9 = dam[(dam.index >= "2026-05-09") & (dam.index < "2026-05-10")]
print("May 9, 2026 prices:")
print(may_9.to_string())
print()
print(f"Total rows: {len(may_9)}")
print(f"Zero hours: {(may_9['price_eur_mwh'] < 0.1).sum()}")
print(f"Negative hours: {(may_9['price_eur_mwh'] < 0).sum()}")
print(f"Min: {may_9['price_eur_mwh'].min():.2f}")

May 9, 2026 prices:
                           price_eur_mwh
2026-05-09 00:00:00+00:00       168.5400
2026-05-09 01:00:00+00:00       147.2000
2026-05-09 02:00:00+00:00       173.5400
2026-05-09 03:00:00+00:00       172.7900
2026-05-09 04:00:00+00:00       146.5125
2026-05-09 05:00:00+00:00       123.3950
2026-05-09 06:00:00+00:00       106.0150
2026-05-09 07:00:00+00:00        33.2900
2026-05-09 08:00:00+00:00         0.0000
2026-05-09 09:00:00+00:00         0.0025
2026-05-09 10:00:00+00:00         0.0000
2026-05-09 11:00:00+00:00         0.0000
2026-05-09 12:00:00+00:00         0.0000
2026-05-09 13:00:00+00:00         0.0000
2026-05-09 14:00:00+00:00        29.9700
2026-05-09 15:00:00+00:00       101.7750
2026-05-09 16:00:00+00:00       146.3625
2026-05-09 17:00:00+00:00       187.5400
2026-05-09 18:00:00+00:00       216.9450
2026-05-09 19:00:00+00:00       185.7625
2026-05-09 20:00:00+00:00       164.8000
2026-05-09 21:00:00+00:00       174.3050
2026-05-09 22:00:00+00:00       162.0

In [8]:
raw_dam = read_csv_from_blob("sagreekdamdevweu", "raw", "dam_prices/historical_2023-01-01_to_2026-05-10.csv")
# Replace <the_date> with whatever's actually there

raw_may_9 = raw_dam[(raw_dam.index >= "2026-05-09") & (raw_dam.index < "2026-05-10")]
print("Raw May 9 prices:")
print(raw_may_9)

Raw May 9 prices:
                           price_eur_mwh
2026-05-09 00:00:00+00:00         168.79
2026-05-09 00:15:00+00:00         169.79
2026-05-09 00:30:00+00:00         167.79
2026-05-09 00:45:00+00:00         167.79
2026-05-09 01:00:00+00:00         145.23
...                                  ...
2026-05-09 22:45:00+00:00         152.49
2026-05-09 23:00:00+00:00         169.94
2026-05-09 23:15:00+00:00         149.29
2026-05-09 23:30:00+00:00         144.58
2026-05-09 23:45:00+00:00         129.70

[96 rows x 1 columns]


In [1]:
may_9

NameError: name 'may_9' is not defined

In [9]:
raw_may_9.resample("H").mean()

C:\Users\HarrisDeralas\AppData\Local\Temp\ipykernel_11260\1558273679.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  raw_may_9.resample("H").mean()


,price_eur_mwh
2026-05-09 00:00:00+00:00,168.5400
2026-05-09 01:00:00+00:00,147.2000
2026-05-09 02:00:00+00:00,173.5400
2026-05-09 03:00:00+00:00,172.7900
2026-05-09 04:00:00+00:00,146.5125
2026-05-09 05:00:00+00:00,123.3950
2026-05-09 06:00:00+00:00,106.0150
2026-05-09 07:00:00+00:00,33.2900
2026-05-09 08:00:00+00:00,0.0000
2026-05-09 09:00:00+00:00,0.0025
